# Lab 05 – Naïve Bayes Classifier
## Lab Task 1: Wine dataset – Gaussian vs Multinomial Naïve Bayes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Load dataset
wine = load_wine()
X, y = wine.data, wine.target
print("Features shape:", X.shape)
print("Classes:", wine.target_names)
print("Feature names:", wine.feature_names)
pd.DataFrame(X, columns=wine.feature_names).head()

In [ ]:
# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
# 3. Train Gaussian NB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_g = gnb.predict(X_test)
acc_g = accuracy_score(y_test, y_pred_g)

# 4. Train Multinomial NB (needs non-negative features - wine features are all >= 0)
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred_m = mnb.predict(X_test)
acc_m = accuracy_score(y_test, y_pred_m)

print(f"Gaussian NB accuracy    : {acc_g*100:.2f}%")
print(f"Multinomial NB accuracy : {acc_m*100:.2f}%")

In [ ]:
# 5. Detailed evaluation
print("=== Gaussian NB ===")
print(classification_report(y_test, y_pred_g, target_names=wine.target_names))
print("=== Multinomial NB ===")
print(classification_report(y_test, y_pred_m, target_names=wine.target_names))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, pred, title in zip(ax, [y_pred_g, y_pred_m], ["Gaussian NB", "Multinomial NB"]):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=wine.target_names, yticklabels=wine.target_names, ax=a, cbar=False)
    a.set_title(title); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); plt.show()

plt.bar(["Gaussian NB", "Multinomial NB"], [acc_g*100, acc_m*100], color=["steelblue", "orange"])
plt.ylabel("Accuracy (%)"); plt.title("Model comparison"); plt.ylim(0, 100); plt.show()

better = "Gaussian NB" if acc_g > acc_m else ("Multinomial NB" if acc_m > acc_g else "Both equal")
print("Better performing model:", better)

In [ ]:
# 6. Predictions on test data using the trained (better) model
results = pd.DataFrame({
    "Actual": [wine.target_names[i] for i in y_test],
    "Gaussian_Pred": [wine.target_names[i] for i in y_pred_g],
    "Multinomial_Pred": [wine.target_names[i] for i in y_pred_m],
})
display(results.head(10))

# Predict a single sample
sample = X_test[0].reshape(1, -1)
print("\nSingle sample -> Gaussian :", wine.target_names[gnb.predict(sample)[0]])
print("Single sample -> Multinomial:", wine.target_names[mnb.predict(sample)[0]])
print("Actual class:", wine.target_names[y_test[0]])
print("Gaussian class probabilities:", np.round(gnb.predict_proba(sample)[0], 3))

**Conclusion (Task 1):** Gaussian NB assumes features follow a normal (continuous) distribution, which suits the wine data (continuous chemical measurements). Multinomial NB is designed for count/frequency data (e.g. word counts), so it generally performs worse here. Check the accuracies printed above to confirm.